In [28]:
import sys, random, importlib
sys.path.insert(0, "..")
import torch

# Reload to pick up any in-session changes to sorl_trainer
import sorl.sorl_trainer as _st; importlib.reload(_st)
from sorl.sorl_trainer import sorl_search, infer_insert_mask, insert_tokens_with_padding
from sorl.trainer_ablate import _drop_nl_prefix_m_set
from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from data.pt_dataset import get_dataset, collate_fn

# ── Config ──────────────────────────────────────────────────────────────
MODEL_NAME  = "Qwen/Qwen3-0.6B"
ABS_VOCAB   = 32
K           = 4
N_SAMPLES   = 4
ANSWER_TOK  = 820   # "####" delimiter in GSM8K

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = SorlModelWrapper.from_pretrained(MODEL_NAME, abstract_vocab_size_list=[ABS_VOCAB])
model     = model.to(device).eval()

base_vocab = int(model.vocab_sizes[0].item())
pad_id     = tokenizer.pad_token_id

# ── GSM8K batch ──────────────────────────────────────────────────────────
ds         = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
batch      = collate_fn([ds[i] for i in range(N_SAMPLES)])
input_ids  = batch["input_ids"].to(device)
attn_mask  = batch["attention_mask"].to(device)
prompt_len = batch["prompt_len"].to(device)

# ── Decode helper ─────────────────────────────────────────────────────────
def decode_annotated(ids_1d, valid_len=None):
    """NL tokens → text, abstract tokens → [ABS]. Pass a 1-D id tensor."""
    n = valid_len if valid_len is not None else len(ids_1d)
    parts, buf = [], []
    for tid in ids_1d[:n].tolist():
        if tid >= base_vocab:
            if buf: parts.append(tokenizer.decode(buf, skip_special_tokens=False)); buf = []
            parts.append("[ABS]")
        else:
            buf.append(tid)
    if buf: parts.append(tokenizer.decode(buf, skip_special_tokens=False))
    return "".join(parts)

print(f"device={device}  base_vocab={base_vocab}  abs_vocab={ABS_VOCAB}  K={K}")
print(f"GSM8K lens: {[int(attn_mask[b].sum()) for b in range(N_SAMPLES)]}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


device=cpu  base_vocab=151936  abs_vocab=32  K=4
GSM8K lens: [100, 99, 164, 183]


In [29]:
# ── Kernel 1: cot_only_abs premise ──────────────────────────────────────────
# sorl_search(cot_only_abs=True) produces:
#
#   [Q tokens]  [ABS_1 ABS_2 ... ABS_N]  [nl_cot_1 nl_cot_2 ... nl_cot_M]  [#### ans]
#               ^^^^^^^^^^^^^^^^^^^^^^^^  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#                  abs prefix block           full NL CoT (untouched)
#
# Internally: abstract tokens are found via recursion in the CoT region,
# then move_abs_to_cot_prefix() collects them all to the CoT start.
# No ABS in query.  No ABS in answer (after ####).

with torch.no_grad():
    best_data, _, _, exp_attn, exp_pl = sorl_search(
        model, input_ids, attn_mask, prompt_len, pad_id,
        n=2, K=K, max_iterations=2,
        memory_span_abs=1792, memory_span_traj=1792,
        temperature=1.0,
        cot_only_abs=True,
    )

print("=== sorl_search(cot_only_abs=True):  [Q] [ABS×N] [nl_cot] [#### ans] ===\n")
for b in range(N_SAMPLES):
    valid = int(exp_attn[b].sum())
    pl    = exp_pl[b].item()
    seq   = best_data[b, :valid]
    resp  = seq[pl:]

    ans_pos = (resp == ANSWER_TOK).nonzero(as_tuple=True)[0]
    ai      = ans_pos[0].item() if len(ans_pos) else len(resp)
    cot     = resp[:ai]
    ans     = resp[ai:]

    n_abs_q   = (seq[:pl] >= base_vocab).sum().item()
    n_abs_cot = (cot >= base_vocab).sum().item()
    n_nl_cot  = (cot <  base_vocab).sum().item()
    n_abs_ans = (ans >= base_vocab).sum().item()

    q_text   = tokenizer.decode([t for t in seq[:pl].tolist() if t < base_vocab], skip_special_tokens=True)
    cot_text = decode_annotated(cot)
    ans_text = tokenizer.decode([t for t in ans.tolist() if t < base_vocab], skip_special_tokens=False)

    print(f"[{b}]  total={valid}  |  ABS: query={n_abs_q}  CoT={n_abs_cot}  ans={n_abs_ans}  |  nl_cot={n_nl_cot}")
    print(f"  Q  : {q_text[:100]}")
    print(f"  CoT: {cot_text[:200]}")
    print(f"  Ans: {ans_text.strip()[:60]}")
    print()

=== sorl_search(cot_only_abs=True):  [Q] [ABS×N] [nl_cot] [#### ans] ===

[0]  total=113  |  ABS: query=0  CoT=13  ans=0  |  nl_cot=54
  Q  : Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in 
  CoT: [ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS] Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.

  Ans: #### 72

[1]  total=113  |  ABS: query=0  CoT=14  ans=0  |  nl_cot=60
  Q  : Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting.
  CoT: [ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS] Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.

  Ans: #### 10

[2]  total=188  |  ABS: query=0  CoT=24  ans=0  |  nl_cot=97
  Q  : Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money 
  CoT:

In [ ]:
# ── Kernel 2: NL Replacement on the prefix-abs layout ───────────────────────
# Input  (from kernel 1):
#   [Q]  [ABS_1 ... ABS_N]  [nl_1 nl_2 ... nl_M]  [#### ans]
#
# _drop_nl_prefix_m_set samples m from M_SET, drops the first m NL tokens:
#
#   [Q]  [ABS_1 ... ABS_N]  [nl_{m+1} ... nl_M]  [#### ans]
#        ^^^^^^^^^^^^^^^^    ^^^^^^^^^^^^^^^^^^^
#         abs block intact    suffix NL only
#
# ABS tokens survive because they sit before the NL block (≥ base_vocab → skipped).
# Answer region is never touched.

import random
random.seed(42)
M_SET = (0, 16, 32, 64, 128)

print(f"M_SET = {M_SET}  (m sampled per-sequence independently)\n")

for trial in range(4):
    print(f"{'─'*72}  trial {trial + 1}")
    out_ids, out_attn, out_pl = _drop_nl_prefix_m_set(
        best_data, exp_attn, exp_pl,
        base_vocab, pad_id, m_set=M_SET, answer_token_id=ANSWER_TOK,
    )
    for b in range(N_SAMPLES):
        valid = int(out_attn[b].sum())
        pl    = out_pl[b].item()
        resp  = out_ids[b, pl:valid]

        ans_pos = (resp == ANSWER_TOK).nonzero(as_tuple=True)[0]
        ai      = ans_pos[0].item() if len(ans_pos) else len(resp)
        cot     = resp[:ai]
        ans     = resp[ai:]

        n_abs = (cot >= base_vocab).sum().item()
        n_nl  = (cot <  base_vocab).sum().item()
        n_abs_ans = (ans >= base_vocab).sum().item()

        cot_text = decode_annotated(cot)
        ans_text = tokenizer.decode([t for t in ans.tolist() if t < base_vocab],
                                    skip_special_tokens=False)

        print(f"  [{b}]  {int(exp_attn[b].sum())}→{valid}tok  |  CoT: {n_abs} abs + {n_nl} nl  |  ABS_ans={n_abs_ans}")
        print(f"       CoT: {cot_text[:180]}")
        print(f"       Ans: {ans_text.strip()[:50]}")
    print()

In [ ]:
# ── Option 1: Free-form generation ──────────────────────────────────────────
# Training layout:  [Q] [ABS×N] [NL_CoT] [#### ans]   (cot_only_abs=True)
# Eval  (this cell): model.generate(free_form=True)
#   → no forced ABS positions; model generates ABS prefix + NL CoT naturally
#   → if the model has learned the prefix format, it produces [ABS×?][NL_CoT][####][ans]
#   → if not yet trained, it mostly produces NL (random init)
#
# This is Option 1: unconstrained — the model controls how many ABS tokens to emit.

QUERY_ONLY = input_ids  # just prompts from the GSM8K batch

with torch.no_grad():
    gen_ff = model.generate(
        QUERY_ONLY, max_new_tokens=120,
        attention_mask=attn_mask,
        temperature=0.0,
        free_form=True,         # ← Option 1: no forced positions
        K=None,
    )

print("=== Option 1: free_form=True  (model controls ABS output) ===\n")
for b in range(N_SAMPLES):
    pl   = prompt_len[b].item()
    resp = gen_ff[b, pl:]

    ans_pos = (resp == ANSWER_TOK).nonzero(as_tuple=True)[0]
    ai      = ans_pos[0].item() if len(ans_pos) else len(resp)
    cot     = resp[:ai]
    ans     = resp[ai:]

    n_abs = (cot >= base_vocab).sum().item()
    n_nl  = (cot <  base_vocab).sum().item()
    cot_text = decode_annotated(cot)
    ans_text = tokenizer.decode([t for t in ans.tolist() if t < base_vocab], skip_special_tokens=False)

    print(f"[{b}]  CoT: {n_abs} abs + {n_nl} nl")
    print(f"  {cot_text[:220]}")
    print(f"  Ans: {ans_text.strip()[:60]}")
    print()

In [ ]:
# ── Option 2: Fixed abs prefix (pause-token style) ───────────────────────────
# Training layout:  [Q] [ABS×8] [NL_CoT] [#### ans]   (cot_only_abs=True, abs_prefix_max=8)
# Eval  (this cell): model.generate(abs_prefix_max=8)
#   → Phase 1: model forced to generate exactly 8 ABS tokens (like pause tokens)
#   → Phase 2: model generates NL CoT + answer freely after the 8-token thinking budget
#
# This is Option 2: constrained budget — fixed 8 ABS "thinking" tokens regardless of sequence length.
# Compare with Option 1: here the budget is fixed, not model-controlled.

ABS_BUDGET = 8

with torch.no_grad():
    gen_prefix = model.generate(
        QUERY_ONLY, max_new_tokens=120,
        attention_mask=attn_mask,
        temperature=0.0,
        abs_prefix_max=ABS_BUDGET,   # ← Option 2: force exactly 8 ABS, then NL
    )

print(f"=== Option 2: abs_prefix_max={ABS_BUDGET}  (fixed ABS budget, then free NL) ===\n")
for b in range(N_SAMPLES):
    pl   = prompt_len[b].item()
    resp = gen_prefix[b, pl:]

    ans_pos = (resp == ANSWER_TOK).nonzero(as_tuple=True)[0]
    ai      = ans_pos[0].item() if len(ans_pos) else len(resp)
    cot     = resp[:ai]
    ans     = resp[ai:]

    n_abs = (cot >= base_vocab).sum().item()
    n_nl  = (cot <  base_vocab).sum().item()
    cot_text = decode_annotated(cot)
    ans_text = tokenizer.decode([t for t in ans.tolist() if t < base_vocab], skip_special_tokens=False)

    print(f"[{b}]  CoT: {n_abs} abs (forced={ABS_BUDGET}) + {n_nl} nl")
    print(f"  {cot_text[:220]}")
    print(f"  Ans: {ans_text.strip()[:60]}")
    print()

M_SET = (0, 16, 32, 64, 128)  (m sampled per-sequence each trial)

────────────────────────────────────────────────────────────────────────  trial 1
  [0]  113→113tok  ABS_cot=13  ABS_ans=0
       CoT:  Natalia sold [ABS]48/2[ABS] = <<48[ABS]/2=2[ABS]4>>24[ABS] clips in May.
[ABS]Natalia sold[ABS] 48+[ABS]24 = <<[ABS]48+2[ABS]4=72[ABS]>>72 clips[ABS] altogether in April and[ABS] 
       Ans: #### 72
  [1]  113→113tok  ABS_cot=14  ABS_ans=0
       CoT:  Weng earns [ABS]12/6[ABS]0 = $<<[ABS]12/6[ABS]0=0.[ABS]2>>0.[ABS]2 per minute.
[ABS]Working 50[ABS] minutes, she earned[ABS] 0.2[ABS] x 50[ABS] = $<<0[ABS].2*5[ABS]0=10[ABS]>>10.

       Ans: #### 10
  [2]  188→156tok  ABS_cot=24  ABS_ans=0
       CoT: [ABS][ABS][ABS][ABS][ABS][ABS][ABS][ABS] grandparents gave her [ABS]15 * [ABS]2 = $<<[ABS]15*2[ABS]=30>>[ABS]30.
This[ABS] means, Betty needs[ABS] 100[ABS] - 50[ABS] - 30[ABS] - 15
       Ans: #### 5
  [3]  213→197tok  ABS_cot=30  ABS_ans=0
       CoT: [ABS][ABS][ABS][ABS]24>>2[ABS]4 pages